In [ ]:
# Combine old + new CMC OHLCV scrapes, first offering per token only
#
# Old scrape : cmc_ohlcv_daily_extended_24m.csv  (Feb 13 2024 – Feb 11 2026, 1 101 tokens)
# New scrape : cmc_ohlcv_daily_raw.csv           (Apr 01 2024 – Mar 30 2026, 1 853 tokens)
# TORD input : tord_clean_min.csv                (offering-level, collapsed to first offering)
#
# Outputs
#   tord_first_offering.csv          – one row per token, earliest ico_start kept
#   outputs_cmc/cmc_ohlcv_combined.csv – merged OHLCV, TORD universe only

from pathlib import Path
import pandas as pd

# ===== CONFIG =====
RAW_DIR       = Path("../raw")
PROCESSED_DIR = Path("../processed")

TORD_CLEAN         = PROCESSED_DIR / "tord_clean_min.csv"
OLD_OHLCV          = RAW_DIR       / "cmc_ohlcv_daily_extended_24m.csv"
NEW_OHLCV          = RAW_DIR       / "cmc_ohlcv_daily_raw.csv"
OUT_TORD_FIRST     = PROCESSED_DIR / "tord_first_offering.csv"
OUT_OHLCV_COMBINED = PROCESSED_DIR / "cmc_ohlcv_combined.csv"

In [ ]:
# ── 1. Build first-offering TORD table ────────────────────────────────────────
# Sort by ico_start ascending so drop_duplicates(keep='first') retains the
# earliest offering per cmc_id. Rows with NaT ico_start are pushed to last.

tord = pd.read_csv(TORD_CLEAN, low_memory=False)
tord["ico_start"] = pd.to_datetime(tord["ico_start"], errors="coerce")
tord["cmc_id"]    = pd.to_numeric(tord["Coinmarketcap_identifier"], errors="coerce")
tord = tord[tord["cmc_id"].notna()].copy()
tord["cmc_id"] = tord["cmc_id"].astype(int)

n_before = len(tord)

tord_first = (
    tord.sort_values("ico_start", ascending=True, na_position="last")
        .drop_duplicates(subset="cmc_id", keep="first")
        .reset_index(drop=True)
)

valid_ids = sorted(tord_first["cmc_id"].unique().tolist())

print(f"Tokens with cmc_id in tord_clean_min : {n_before}")
print(f"Unique tokens after first-offering    : {len(tord_first)}")
print(f"Later offerings dropped               : {n_before - len(tord_first)}")

tord_first.to_csv(OUT_TORD_FIRST, index=False)
print(f"\nSaved: {OUT_TORD_FIRST}")

In [ ]:
# ── 2. Load both OHLCV scrapes ────────────────────────────────────────────────

def load_ohlcv(path):
    df = pd.read_csv(path, low_memory=False)
    df["date"]   = pd.to_datetime(df["date"], errors="coerce").dt.date
    df["cmc_id"] = pd.to_numeric(df["cmc_id"], errors="coerce")
    df = df.dropna(subset=["cmc_id", "date"]).copy()
    df["cmc_id"] = df["cmc_id"].astype(int)
    return df

old = load_ohlcv(OLD_OHLCV)
new = load_ohlcv(NEW_OHLCV)

print("Old OHLCV (extended 24m scrape):")
print(f"  {len(old):>10,} rows | {old['cmc_id'].nunique():>5} tokens | {old['date'].min()} → {old['date'].max()}")
print("\nNew OHLCV (final scrape):")
print(f"  {len(new):>10,} rows | {new['cmc_id'].nunique():>5} tokens | {new['date'].min()} → {new['date'].max()}")

In [ ]:
# ── 3. Restrict to TORD universe and combine ──────────────────────────────────
# Stack new on top of old so drop_duplicates(keep='first') favours new data
# for any (cmc_id, date) that appears in both scrapes.

old_f = old[old["cmc_id"].isin(valid_ids)].copy()
new_f = new[new["cmc_id"].isin(valid_ids)].copy()

print(f"Old after universe filter: {len(old_f):>10,} rows | {old_f['cmc_id'].nunique()} tokens")
print(f"New after universe filter: {len(new_f):>10,} rows | {new_f['cmc_id'].nunique()} tokens")

combined = (
    pd.concat([new_f, old_f], ignore_index=True)
      .drop_duplicates(subset=["cmc_id", "date"], keep="first")
      .sort_values(["cmc_id", "date"])
      .reset_index(drop=True)
)

print(f"\nCombined OHLCV:")
print(f"  {len(combined):>10,} rows")
print(f"  {combined['cmc_id'].nunique():>10} tokens")
print(f"  {combined['date'].min()} → {combined['date'].max()}")

In [ ]:
# ── 4. Coverage summary ───────────────────────────────────────────────────────

old_ids  = set(old_f["cmc_id"].unique())
new_ids  = set(new_f["cmc_id"].unique())
both     = old_ids & new_ids
new_only = new_ids - old_ids
old_only = old_ids - new_ids

print(f"Tokens in BOTH scrapes  → full Feb 2024 – Mar 2026 : {len(both):>5}")
print(f"Tokens in NEW only      → Apr 2024 – Mar 2026      : {len(new_only):>5}")
print(f"Tokens in OLD only      → Feb 2024 – Feb 2026      : {len(old_only):>5}")
print(f"\nTotal tokens in combined dataset                   : {combined['cmc_id'].nunique():>5}")

In [ ]:
# ── 5. Save ───────────────────────────────────────────────────────────────────

combined.to_csv(OUT_OHLCV_COMBINED, index=False)
print(f"Saved: {OUT_OHLCV_COMBINED}")
print(f"Size : {OUT_OHLCV_COMBINED.stat().st_size / 1e6:.1f} MB")